# TMS-EEG masking-noise planner and player

A prototyping tool for mixing probabilistic click events into either **white noise** or **click-spectrum-shaped noise**. Use isolated **Single pulse** events or **Rhythmic** trains with adjustable frequency and pulses per train.

> **Safety boundary:** digital amplitude is not dB SPL. This notebook is not a calibrated level controller, TMS trigger, medical device, or complete auditory/somatosensory control. Start hardware volume low, use an approved acoustic calibration and hearing-protection procedure, and follow institutional and device safety rules. Never infer safe exposure from the slider value.

Learning goals: generate a masking candidate, preview it cautiously, document the setup, and decide whether the masking/control setup needs retuning. Requires `numpy`, `ipywidgets`, and `IPython`.

## 1. Setup and click recording

The notebook uses `assets/singlepulse.wav` when present. It extracts one transient around that recording's strongest peak, uses that same transient both to shape the click-derived noise spectrum and to render scheduled clicks, and leaves the 4-Hz/20-Hz train recordings unused. Uncompressed 16- or 24-bit PCM mono/stereo WAV is supported. Retune masking whenever the coil, intensity, placement/orientation, foam, room, microphone geometry, or transducer changes.

In [3]:
from pathlib import Path
import base64
import json
import math
import sys

import ipywidgets as widgets
from IPython.display import HTML, Javascript, display

candidates = [Path.cwd(), Path.cwd() / 'output' / 'jupyter-notebook']
HERE = next((p for p in candidates if (p / 'masking_audio.py').is_file()), None)
if HERE is None:
    raise FileNotFoundError('Run from the notebook directory or repository root.')
sys.path.insert(0, str(HERE))
from masking_audio import DEFAULT_SEED, create_click_preview_wav, create_masking_wav, make_fake_single_click, schedule_click_times

RECORDED_CLICK_FILE = HERE / 'assets' / 'singlepulse.wav'
FALLBACK_CLICK_FILE = HERE / 'assets' / 'fake_single_tms_click.wav'
if not FALLBACK_CLICK_FILE.exists():
    make_fake_single_click(FALLBACK_CLICK_FILE)
CLICK_FILE = RECORDED_CLICK_FILE if RECORDED_CLICK_FILE.exists() else FALLBACK_CLICK_FILE
print(f'Single-pulse source for noise shaping and mixing: {CLICK_FILE}')

Single-pulse source for noise shaping and mixing: /Users/nikolaj_syrov/Documents/Python_proj/RUB/mics/CAMpy-TMS/assets/singlepulse.wav


## 2. Playback controls

Choose **White noise** or **Click-shaped noise**, then choose **Single pulse** or **Rhythmic**. Single-pulse opportunities are jittered. In rhythmic mode, frequency controls spacing inside a train, pulses per train is exact, and click probability selects complete trains. `Click volume` is a relative click-to-noise control and reaches 50; increasing it makes clicks dominate the noise but does not exceed the master digital peak. Use **Test one click** before starting the full masker. Duration is in seconds (maximum 300). Master amplitude reaches `1.00` (0 dBFS, the unclipped digital maximum); it is not dB SPL. Infinite playback uses Web Audio to loop a periodic noise buffer without restarting an HTML media element; crossing click tails wrap to the beginning. **Upload settings** opens the local `generated` folder on macOS, with browser upload as fallback. These are acoustic previews only—not rTMS trigger or dosing controls.

In [4]:
wide = widgets.Layout(width='850px')
fine_style = {'description_width': '160px'}
duration_seconds = widgets.FloatSlider(value=30.0, min=0.5, max=300.0, step=0.5, readout_format='.1f', description='Buffer duration (s)', continuous_update=False, layout=wide, style=fine_style)
volume = widgets.FloatSlider(value=0.05, min=0.0, max=1.0, step=0.01, readout_format='.2f', description='Master amplitude', continuous_update=False, layout=wide, style=fine_style)
noise_mode = widgets.Dropdown(options=[('White noise', 'white'), ('Click-shaped noise', 'click-shaped')], value='click-shaped', description='Noise type')
include_clicks = widgets.Checkbox(value=True, description='Include scheduled clicks', indent=False)
click_volume = widgets.FloatSlider(value=5.0, min=0.0, max=50.0, step=0.5, description='Click volume', continuous_update=False, layout=wide, style=fine_style)
click_schedule = widgets.Dropdown(options=[('Single pulse', 'single-pulse'), ('Rhythmic', 'rhythmic')], value='single-pulse', description='Schedule')
click_probability = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.01, readout_format='.2f', description='Pulse/train probability', continuous_update=False, layout=wide, style=fine_style)
single_pulse_rate = widgets.FloatSlider(value=1.0, min=0.1, max=10.0, step=0.05, description='Single-pulse opp./s', continuous_update=False, layout=wide, style=fine_style)
rhythmic_frequency = widgets.FloatSlider(value=5.0, min=0.5, max=20.0, step=0.1, description='Rhythmic frequency (Hz)', continuous_update=False, layout=wide, style=fine_style)
pulses_per_train = widgets.IntSlider(value=5, min=1, max=100, step=1, description='Pulses per train', continuous_update=False, layout=wide, style=fine_style)
inter_train_interval = widgets.FloatSlider(value=1.0, min=0.0, max=30.0, step=0.1, description='Inter-train interval (s)', continuous_update=False, layout=wide, style=fine_style)
infinite_loop = widgets.Checkbox(value=False, description='Play infinitely (loop generated buffer)', indent=False)
acknowledge = widgets.Checkbox(value=False, description='I understand amplitude is not dB SPL; hardware volume starts low', indent=False)
quieter = widgets.Button(description='− volume', button_style='info')
louder = widgets.Button(description='+ volume', button_style='warning')
test_click = widgets.Button(description='Test one click', button_style='warning', icon='volume-up', disabled=True)
play = widgets.Button(description='Play / rebuild', button_style='success', icon='play', disabled=True)
pause = widgets.Button(description='Pause', icon='pause')
playback_indicator = widgets.HTML(value='<div id="masking-playback-indicator" style="padding:12px 18px;border-radius:8px;background:#555;color:white;font-size:20px;font-weight:700;text-align:center">● STOPPED</div>')
status = widgets.HTML(value='<em>Playback locked pending acknowledgement.</em>')
level_warning = widgets.HTML()
click_warning = widgets.HTML()
player = widgets.Output()
save_settings = widgets.Button(description='Save settings', icon='download')
settings_upload = widgets.FileUpload(accept='.json,application/json', multiple=False, description='Browser upload')
choose_settings = widgets.Button(description='Upload settings', icon='folder-open')
selected_settings_path = widgets.Text(value='', description='Selected file', disabled=True, layout=wide, style=fine_style)
apply_settings = widgets.Button(description='Apply uploaded settings', icon='upload')
settings_output = widgets.Output()

def change_volume(delta):
    volume.value = min(volume.max, max(volume.min, round(volume.value + delta, 3)))

quieter.on_click(lambda _: change_volume(-volume.step))
louder.on_click(lambda _: change_volume(volume.step))

def unlock(change):
    play.disabled = not change['new']
    test_click.disabled = not change['new']
    status.value = '<em>Ready; verify hardware level and calibration.</em>' if change['new'] else '<em>Playback locked pending acknowledgement.</em>'
acknowledge.observe(unlock, names='value')

def update_level_warning(_=None):
    if volume.value == 0:
        level_warning.value = '<span>Muted (digital amplitude 0).</span>'
        return
    peak_dbfs = 20 * math.log10(volume.value)
    if volume.value > 0.5:
        level_warning.value = f'<b style="color:#b00020">HIGH DIGITAL LEVEL: peak {peak_dbfs:.1f} dBFS. This is not dB SPL; re-check calibration and exposure limits.</b>'
    else:
        level_warning.value = f'<span>Digital peak: {peak_dbfs:.1f} dBFS (not dB SPL).</span>'

volume.observe(update_level_warning, names='value')
update_level_warning()

def update_click_warning(_=None):
    if click_volume.value > 10:
        click_warning.value = '<b style="color:#b00020">High relative click boost: clicks strongly dominate the masking noise. Re-check comfort and calibrated exposure.</b>'
    else:
        click_warning.value = '<span>Click volume controls click-to-noise ratio, not absolute dB SPL.</span>'

click_volume.observe(update_click_warning, names='value')
update_click_warning()

def update_schedule_controls(_=None):
    enabled = include_clicks.value
    single_pulse_rate.disabled = not enabled or click_schedule.value != 'single-pulse'
    rhythmic_frequency.disabled = not enabled or click_schedule.value != 'rhythmic'
    pulses_per_train.disabled = not enabled or click_schedule.value != 'rhythmic'
    inter_train_interval.disabled = not enabled or click_schedule.value != 'rhythmic'
    click_probability.disabled = not enabled
    click_volume.disabled = not enabled

include_clicks.observe(update_schedule_controls, names='value')
click_schedule.observe(update_schedule_controls, names='value')
update_schedule_controls()

def on_test_click(_):
    output_file = HERE / 'generated' / 'single_click_test.wav'
    create_click_preview_wav(CLICK_FILE, output_file, volume=volume.value)
    encoded = base64.b64encode(output_file.read_bytes()).decode('ascii')
    playback_indicator.value = '<div id="masking-playback-indicator" style="padding:12px 18px;border-radius:8px;background:#d97706;color:white;font-size:20px;font-weight:700;text-align:center">● TEST CLICK</div>'
    click_js = f'''
    (async () => {{
      let shared; try {{ shared = window.top; }} catch (error) {{ shared = window; }}
      shared.__stopMaskingAudio?.();
      const Context = window.AudioContext || window.webkitAudioContext;
      const context = shared.__maskingAudioContext || new Context(); shared.__maskingAudioContext = context; await context.resume();
      const binary = atob('{encoded}'); const bytes = new Uint8Array(binary.length);
      for (let i=0; i<binary.length; i++) bytes[i]=binary.charCodeAt(i);
      const buffer = await context.decodeAudioData(bytes.buffer); const source = context.createBufferSource();
      source.buffer=buffer; source.connect(context.destination);
      source.onended=() => {{ const el=document.getElementById('masking-playback-indicator'); if(el) {{ el.innerHTML='● STOPPED'; el.style.background='#555'; }} }};
      source.start(0); shared.__maskingPlayer={{context,source,buffer}};
    }})().catch(error => console.error('Click test failed:',error));
    '''
    with player: player.clear_output(wait=True); display(Javascript(click_js))
    status.value = f'<b>Testing one extracted click</b> at digital peak {volume.value:.3f}; not dB SPL.'

def on_play(_):
    kind = 'preview_with_clicks' if include_clicks.value else 'masking_candidate'
    output_file = HERE / 'generated' / f'{kind}.wav'
    create_masking_wav(CLICK_FILE, output_file, duration_s=duration_seconds.value, volume=volume.value, noise_mode=noise_mode.value, include_clicks=include_clicks.value, click_volume=click_volume.value, click_schedule=click_schedule.value, single_pulse_rate_hz=single_pulse_rate.value, rhythmic_frequency_hz=rhythmic_frequency.value, pulses_per_train=pulses_per_train.value, inter_train_interval_s=inter_train_interval.value, click_probability=click_probability.value)
    encoded = base64.b64encode(output_file.read_bytes()).decode('ascii')
    loop_value = 'true' if infinite_loop.value else 'false'
    playback_indicator.value = '<div id="masking-playback-indicator" style="padding:12px 18px;border-radius:8px;background:#0a8f3c;color:white;font-size:20px;font-weight:700;text-align:center;animation:maskPulse 1s infinite alternate"><style>@keyframes maskPulse{{from{{opacity:1}}to{{opacity:.45}}}}</style>▶ MASKING PLAYING</div>'
    web_audio_js = f'''
    (async () => {{
      let shared;
      try {{ shared = window.top; }} catch (error) {{ shared = window; }}
      shared.__stopMaskingAudio = () => {{
        shared.__maskingGeneration = (shared.__maskingGeneration || 0) + 1;
        if (shared.__maskingPlayer?.source) {{
          try {{ shared.__maskingPlayer.source.stop(); }} catch (error) {{}}
        }}
        shared.__maskingPlayer = null;
      }};
      shared.__stopMaskingAudio();
      const generation = shared.__maskingGeneration;
      const Context = window.AudioContext || window.webkitAudioContext;
      const context = shared.__maskingAudioContext || new Context();
      shared.__maskingAudioContext = context;
      await context.resume();
      const binary = atob('{encoded}');
      const bytes = new Uint8Array(binary.length);
      for (let i = 0; i < binary.length; i++) bytes[i] = binary.charCodeAt(i);
      const buffer = await context.decodeAudioData(bytes.buffer);
      if (shared.__maskingGeneration !== generation) return;
      const source = context.createBufferSource();
      source.buffer = buffer;
      source.loop = {loop_value};
      source.connect(context.destination);
      source.onended = () => {{
        if (shared.__maskingGeneration === generation && !source.loop) {{
          const el=document.getElementById('masking-playback-indicator');
          if(el) {{ el.innerHTML='● FINISHED'; el.style.background='#555'; el.style.animation='none'; }}
        }}
      }};
      source.start(0);
      shared.__maskingPlayer = {{context, source, buffer}};
    }})().catch(error => console.error('Masking Web Audio failed:', error));
    '''
    with player:
        player.clear_output(wait=True)
        display(Javascript(web_audio_js))
        display(HTML("<button onclick='window.top.__maskingAudioContext?.resume()'>Resume Web Audio if playback was blocked</button> <button onclick='window.top.__stopMaskingAudio?.()'>Emergency stop</button>"))
    if include_clicks.value:
        times = schedule_click_times(duration_seconds.value, click_schedule.value, single_pulse_rate.value, rhythmic_frequency.value, pulses_per_train.value, inter_train_interval.value, click_probability.value, DEFAULT_SEED + 1)
        event_text = f'{len(times)} clicks ({click_schedule.label}; complete trains in rhythmic mode)'
    else:
        event_text = 'masking only'
    loop_text = 'infinite loop' if infinite_loop.value else 'one pass'
    status.value = f'<b>Playing:</b> {event_text}; {duration_seconds.value:.1f} s buffer, {loop_text}, digital peak {volume.value:.3f}; not dB SPL.'

def on_pause(_):
    pause_js = "let shared; try { shared = window.top; } catch (error) { shared = window; } if (shared.__stopMaskingAudio) { shared.__stopMaskingAudio(); } else { shared.__maskingGeneration = (shared.__maskingGeneration || 0) + 1; if (shared.__maskingPlayer?.source) { try { shared.__maskingPlayer.source.stop(); } catch (error) {} shared.__maskingPlayer = null; } }"
    with player:
        display(Javascript(pause_js))
    playback_indicator.value = '<div id="masking-playback-indicator" style="padding:12px 18px;border-radius:8px;background:#555;color:white;font-size:20px;font-weight:700;text-align:center">● STOPPED</div>'
    status.value = '<b>Stopped.</b> Press Play/rebuild to start from the buffer beginning.'

settings_widgets = {
    'duration_seconds': duration_seconds, 'master_amplitude': volume,
    'noise_mode': noise_mode, 'include_clicks': include_clicks,
    'click_volume': click_volume, 'click_schedule': click_schedule,
    'click_probability': click_probability, 'single_pulse_rate_hz': single_pulse_rate,
    'rhythmic_frequency_hz': rhythmic_frequency, 'pulses_per_train': pulses_per_train,
    'inter_train_interval_s': inter_train_interval, 'infinite_loop': infinite_loop,
}

def current_settings():
    return {'schema_version': 2, **{name: widget.value for name, widget in settings_widgets.items()}}

def filename_number(value):
    return f'{value:g}'.replace('-', 'm').replace('.', 'p')

def settings_filename():
    base = [
        'masking', noise_mode.value, click_schedule.value,
        f'{filename_number(duration_seconds.value)}s',
        f'prob{filename_number(click_probability.value)}',
        f'amp{filename_number(volume.value)}',
        'clicks-on' if include_clicks.value else 'clicks-off',
        f'cv{filename_number(click_volume.value)}',
    ]
    if click_schedule.value == 'single-pulse':
        base.append(f'rate{filename_number(single_pulse_rate.value)}Hz')
    else:
        base.extend([f'freq{filename_number(rhythmic_frequency.value)}Hz', f'{pulses_per_train.value}pulses', f'iti{filename_number(inter_train_interval.value)}s'])
    base.append('loop' if infinite_loop.value else 'once')
    return '_'.join(base) + '.json'

def on_save_settings(_):
    payload = json.dumps(current_settings(), indent=2).encode('utf-8')
    filename = settings_filename()
    settings_file = HERE / 'generated' / filename
    settings_file.parent.mkdir(parents=True, exist_ok=True)
    settings_file.write_bytes(payload)
    encoded = base64.b64encode(payload).decode('ascii')
    link = f'Saved <code>{filename}</code> locally. <a download="{filename}" href="data:application/json;base64,{encoded}">Download {filename}</a>'
    with settings_output:
        settings_output.clear_output(wait=True)
        display(HTML(link))

def uploaded_bytes(upload):
    value = upload.value
    if not value:
        raise ValueError('Choose a JSON settings file first.')
    item = next(iter(value.values())) if isinstance(value, dict) else value[0]
    return bytes(item['content'])

def on_choose_settings(_):
    from tkinter import Tk, filedialog

    folder = HERE / 'generated'
    folder.mkdir(parents=True, exist_ok=True)
    root = None
    try:
        root = Tk()
        root.withdraw()
        root.attributes('-topmost', True)
        chosen = filedialog.askopenfilename(initialdir=str(folder), title='Choose masking-player settings JSON', filetypes=[('JSON settings', '*.json')])
    except Exception as exc:
        status.value = f'<b>Native chooser unavailable:</b> {exc}. Use browser upload below.'
        return
    finally:
        if root is not None:
            root.destroy()
    if chosen:
        selected_settings_path.value = chosen
        status.value = f'<b>Selected:</b> {selected_settings_path.value}'

def selected_settings_bytes():
    if selected_settings_path.value:
        path = Path(selected_settings_path.value)
        if path.suffix.lower() != '.json' or not path.is_file():
            raise ValueError('Selected settings path must be an existing JSON file.')
        return path.read_bytes()
    return uploaded_bytes(settings_upload)

def validate_setting(widget, value):
    if isinstance(widget, widgets.Checkbox):
        if not isinstance(value, bool):
            raise ValueError('checkbox settings must be true or false')
    elif isinstance(widget, widgets.Dropdown):
        allowed = [option[1] if isinstance(option, tuple) else option for option in widget.options]
        if value not in allowed:
            raise ValueError(f'invalid choice: {value}')
    elif isinstance(widget, widgets.IntSlider):
        if not isinstance(value, int) or not widget.min <= value <= widget.max:
            raise ValueError(f'integer value {value} outside [{widget.min}, {widget.max}]')
    elif not isinstance(value, (int, float)) or not widget.min <= value <= widget.max:
        raise ValueError(f'value {value} outside [{widget.min}, {widget.max}]')

def on_apply_settings(_):
    try:
        data = json.loads(selected_settings_bytes().decode('utf-8'))
        if data.get('schema_version') == 1:
            data['schema_version'] = 2
            data['duration_seconds'] = data.pop('duration_minutes') * 60.0
            data['click_schedule'] = 'single-pulse' if data.get('click_schedule') == 'random' else data.get('click_schedule')
            data['single_pulse_rate_hz'] = data.pop('random_rate_hz')
            data['pulses_per_train'] = 5
            data['inter_train_interval_s'] = 1.0
        if data.get('schema_version') != 2:
            raise ValueError('Unsupported settings schema version.')
        for name, widget in settings_widgets.items():
            if name not in data:
                raise ValueError(f'Missing setting: {name}')
            validate_setting(widget, data[name])
        for name, widget in settings_widgets.items():
            widget.value = data[name]
        acknowledge.value = False
        update_schedule_controls()
        status.value = '<b>Settings loaded.</b> Safety acknowledgement reset; review settings before playback.'
    except Exception as exc:
        status.value = f'<b>Settings load failed:</b> {exc}'

test_click.on_click(on_test_click)
play.on_click(on_play)
pause.on_click(on_pause)
save_settings.on_click(on_save_settings)
choose_settings.on_click(on_choose_settings)
apply_settings.on_click(on_apply_settings)
controls = widgets.VBox([playback_indicator, duration_seconds, volume, level_warning, noise_mode, include_clicks, click_volume, click_warning, click_schedule, click_probability, single_pulse_rate, rhythmic_frequency, pulses_per_train, inter_train_interval, infinite_loop, acknowledge, widgets.HBox([quieter, louder, test_click, play, pause]), widgets.HBox([save_settings, choose_settings, apply_settings]), selected_settings_path, widgets.HTML('<em>Browser upload fallback:</em>'), settings_upload, settings_output, status, player])
display(controls)